In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
root_override = os.environ.get("OCL_PROJECT_ROOT")
candidates = [Path(root_override).resolve()] if root_override else [cwd, *cwd.parents]
project_root = next((p for p in candidates if (p / "src/dataset.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Open the notebook from the OCL workspace or set OCL_PROJECT_ROOT.")

sys.path.insert(0, str(project_root))
data_root = Path(os.environ.get("OCL_DATA_ROOT", project_root / "data"))
resource_root = Path(os.environ.get("OCL_RESOURCE_ROOT", data_root / "resources"))

import torch
from torch.utils.data import DataLoader

from src.cache import get_image_features
from src.dataset import OCLDataset
from src.model import load_model, encode_text
from src.prompts import build_affordance_prompts

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cache_dir = Path(os.environ.get("OCL_CACHE_DIR", data_root / "cache"))
print(device)

In [ ]:
model, preprocess, weights_sha256 = load_model("ViT-B/32", device, os.environ.get("OCL_WEIGHTS_DIR"))
dataset = OCLDataset(data_root, resource_root, preprocess)
test_loader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=0, drop_last=False)

In [ ]:
results, cache_key, cache_hit = get_image_features(
    model, test_loader, device, weights_sha256, use_cache=True, cache_dir=cache_dir
)

image_features = results["features"]
print(image_features.shape)

In [ ]:
prompts_v1 = build_affordance_prompts(dataset.affordances, mode="word")
prompts_v2 = build_affordance_prompts(dataset.affordances, mode="definition")
prompts_v3 = build_affordance_prompts(dataset.affordances, mode="combined")

In [ ]:
print("V1:", len(set(prompts_v1)))
print("V2:", len(set(prompts_v2)))
print("V3:", len(set(prompts_v3)))

In [ ]:
from src.metrics import mean_average_precision
def evaluate_prompts(prompts):

    text_features = encode_text(
        model,
        prompts,
        device
    ).cpu()

    scores = image_features @ text_features.T

    map_score, ap = mean_average_precision(
        scores,
        results["affordance"]
    )

    return map_score, ap

In [ ]:
map_v1, ap_v1 = evaluate_prompts(prompts_v1)
map_v2, ap_v2 = evaluate_prompts(prompts_v2)
map_v3, ap_v3 = evaluate_prompts(prompts_v3)

def percent(value):
    return "undefined" if value is None else f"{value:.2%}"

print(f"V1 Word only:         {percent(map_v1)}")
print(f"V2 Definition only:   {percent(map_v2)}")
print(f"V3 Word + Definition: {percent(map_v3)}")

In [ ]:
from collections import Counter
import numpy as np

prompt_counts = Counter(prompts_v1)

collision_mask = np.array([
    prompt_counts[prompt] > 1
    for prompt in prompts_v1
])

print("Collision labels:", collision_mask.sum())

print(
    "Valid collision labels:",
    (collision_mask & np.isfinite(ap_v1)).sum()
)

In [ ]:
for name, ap in [
    ("V1", ap_v1),
    ("V2", ap_v2),
    ("V3", ap_v3)
]:
    print(name)

    print(
        "Collision mAP:",
        np.nanmean(ap[collision_mask])
    )

    print(
        "Non-collision mAP:",
        np.nanmean(ap[~collision_mask])
    )

In [ ]:
from collections import defaultdict
from itertools import combinations

groups = defaultdict(list)

for i, prompt in enumerate(prompts_v1):
    groups[prompt].append(i)

collision_pairs = []

for ids in groups.values():
    if len(ids) > 1:
        collision_pairs.extend(
            combinations(ids, 2)
        )

print(collision_pairs)
print("Number of pairs:", len(collision_pairs))

In [ ]:
from src.metrics import pairwise_disambiguation

In [ ]:
for name, prompts in [
    ("V1", prompts_v1),
    ("V2", prompts_v2),
    ("V3", prompts_v3)
]:

    text_features = encode_text(
        model, prompts, device
    ).cpu()

    scores = image_features @ text_features.T

    acc, valid_pairs = pairwise_disambiguation(
        scores,
        results["affordance"],
        collision_pairs
    )

    print(
        f"{name}: {percent(acc)}",
        f"| Valid pairs: {valid_pairs}"
    )